# Text Simplification Model – Inference & Analysis
This notebook contains inference and detailed evaluation for a BART-based text simplification model.

In [ ]:

# Imports
import os, re, json
from dataclasses import dataclass
from typing import Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from datasets import load_dataset
from transformers import BartTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import corpus_bleu
from easse.sari import get_corpus_sari_operation_scores
import textstat


In [ ]:

# Configuration
@dataclass
class InferenceConfig:
    model_path: str = "./bart_simplification_scenario_2_improved/best_model"
    max_length: int = 100
    min_length: int = 10
    num_beams: int = 5
    length_penalty: float = 1.0
    no_repeat_ngram_size: int = 3
    early_stopping: bool = True
    output_dir: str = "./inference_results"
    num_best: int = 5
    num_worst: int = 5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

config = InferenceConfig()
os.makedirs(config.output_dir, exist_ok=True)
print(config)


In [ ]:

# Text normalization utilities
class TextProcessor:
    PTB_PATTERNS = [
        (r'\b-?lrb-?\b', '('),
        (r'\b-?rrb-?\b', ')'),
    ]
    @staticmethod
    def normalize_text(text):
        if not text:
            return ""
        for p, r in TextProcessor.PTB_PATTERNS:
            text = re.sub(p, r, text, flags=re.I)
        return re.sub(r'\s+', ' ', text).strip()


In [ ]:

# Load model
tokenizer = BartTokenizer.from_pretrained(config.model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(config.model_path).to(config.device)
model.eval()
print("Model loaded")


In [ ]:

# Load dataset
dataset = load_dataset("bogdancazan/wikilarge-text-simplification")["test"]

def preprocess(ex):
    return {
        "source": TextProcessor.normalize_text(ex["Normal"]),
        "target": TextProcessor.normalize_text(ex["Simple"])
    }

dataset = dataset.map(preprocess)
print("Samples:", len(dataset))


In [ ]:

# Inference
sources, predictions, references = [], [], []

with torch.no_grad():
    for ex in tqdm(dataset):
        inputs = tokenizer(ex["source"], return_tensors="pt", truncation=True).to(config.device)
        out = model.generate(**inputs, max_length=config.max_length, num_beams=config.num_beams)
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        sources.append(ex["source"])
        references.append(ex["target"])
        predictions.append(pred)


In [ ]:

# Metrics
sari_add, sari_keep, sari_del = get_corpus_sari_operation_scores(
    sources, predictions, [references]
)
sari = (sari_add + sari_keep + sari_del) / 3
bleu = corpus_bleu(predictions, [references]).score

print("SARI:", sari)
print("BLEU:", bleu)


In [ ]:

# Readability (FKGL)
src_fkgl = [textstat.flesch_kincaid_grade(s) for s in sources]
pred_fkgl = [textstat.flesch_kincaid_grade(p) for p in predictions]

print("FKGL Source:", np.mean(src_fkgl))
print("FKGL Prediction:", np.mean(pred_fkgl))


In [ ]:

# Visualization
plt.hist([sari]*len(predictions), bins=10)
plt.title("SARI Distribution")
plt.show()
